[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/20_embedding_solution.ipynb)

# 🟢 Solution: Embedding Layer

*Core Ops & Layers · Easy*

Reference implementation. Try it yourself in `20_embedding.ipynb` first.

---
Implement an **embedding table** as an `nnx.Module`.

Map integer token ids to dense vectors: `(...) -> (..., features)`.

### Rules
- Signature: `MyEmbedding(num_embeddings, features, *, rngs)`
- Table stored as `self.table`, an `nnx.Param` of shape `(num_embeddings, features)`
- Initialise with `jax.random.normal(...) * 0.02` (the GPT-2 convention)
- `__call__(ids)` accepts **any** shape of integer ids
- Also implement `attend(x)`: `(..., features) -> (..., num_embeddings)`,
  the transpose projection used for weight tying

### Indexing vs one-hot
These compute the same thing:

```python
table[ids]                        # gather
jax.nn.one_hot(ids, V) @ table    # matmul
```

The gather is `O(1)` per token; the matmul is `O(V)` per token and materialises a
`(B, T, V)` intermediate — with `V = 50257` that is enormous. Always gather.

(The one-hot form is not useless, though: on TPU it can be faster for small
vocabularies, and it is how you'd explain the *gradient*.)

### Why the gradient is sparse
`d(loss)/d(table)` is nonzero only at the rows you actually looked up, and
repeated ids **accumulate**. JAX handles this correctly through
`.at[].add()` semantics under the hood — but it produces a *dense* gradient array
with mostly zeros, which is why large-vocabulary models want sparse optimizer
support.

### Weight tying
`attend` exists because most language models share one matrix between the input
embedding and the output projection. It saves `V x d` parameters (about 40M for
GPT-2) and generally improves perplexity.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp
from flax import nnx


class MyEmbedding(nnx.Module):
    def __init__(self, num_embeddings: int, features: int, *, rngs: nnx.Rngs):
        key = rngs.params()
        self.table = nnx.Param(
            jax.random.normal(key, (num_embeddings, features)) * 0.02
        )
        self.num_embeddings = num_embeddings
        self.features = features

    def __call__(self, ids):
        # A gather, not a one-hot matmul. Advanced indexing already handles
        # arbitrary leading shapes.
        return self.table[ids]

    def attend(self, x):
        # Weight tying: reuse the same matrix for the output projection.
        return x @ self.table[...].T

In [ ]:
# 🔍 Verify
import jax.numpy as jnp
from flax import nnx

emb = MyEmbedding(100, 8, rngs=nnx.Rngs(params=0))

print("table:", emb.table.shape)
print("scalar id  ->", emb(jnp.array(5)).shape)
print("(3,) ids   ->", emb(jnp.array([1, 2, 3])).shape)
print("(2,4) ids  ->", emb(jnp.zeros((2, 4), dtype=jnp.int32)).shape)
print("attend     ->", emb.attend(jnp.ones((2, 4, 8))).shape)

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("embedding")